# CUAD retained evidence: scoring grain and replay

CG-25 recovery, 2026-09-09. This notebook reruns offline checks over the preserved files. No API calls. The full original extraction trace and resolved model settings remain unavailable.


In [1]:
from pathlib import Path
import importlib.util, json, sys
sys.dont_write_bytecode = True
root = Path.cwd()
if not (root / "replay.py").exists():
    root = root / "fixtures/semantic-neurons/cuad-2026-07-22"
spec = importlib.util.spec_from_file_location("cuad_replay", root / "replay.py")
replay = importlib.util.module_from_spec(spec)
spec.loader.exec_module(replay)
result = replay.replay()
print("All preserved artifact digests verified.")


All preserved artifact digests verified.


## Dataset grain

Predictions are retained accepted-evidence rows. Gold entries are annotation-list elements within a contract/category. A category can have multiple gold entries; these units cannot be mixed.


In [2]:
for split, data in result["splits"].items():
    print(split, json.dumps(data["profile"], indent=2))


design {
  "contracts": 10,
  "chunks": 702,
  "categories": 8,
  "gold_entries": 132,
  "positive_contract_categories": 55,
  "exact_duplicate_gold_entries_within_category": 0,
  "prediction_rows": 62,
  "contracts_without_predictions": [
    "GluMobileInc_20070319_S-1A_EX-10.09_436630_EX-10.09_Content License Agreement4"
  ]
}
holdout {
  "contracts": 40,
  "chunks": 1705,
  "categories": 8,
  "gold_entries": 330,
  "positive_contract_categories": 170,
  "exact_duplicate_gold_entries_within_category": 0,
  "prediction_rows": 164,
  "contracts_without_predictions": [
    "TELKOMSALTD_01_30_2003-EX-10-LICENCE AND MAINTENANCE AGREEMENT",
    "GridironBionutrientsInc_20171206_8-K_EX-10.2_10972556_EX-10.2_Endorsement Agreement",
    "MFAFINANCIAL,INC_07_06_2020-EX-99.D-JOINT FILING AGREEMENT",
    "XLITECHNOLOGIES,INC_12_11_2015-EX-10.1-Sponsorship Agreement"
  ]
}


## Legacy versus consistent counts

Matching and input order are unchanged. The legacy scorer counts at most one false negative for a wholly missed contract/category; the correction counts every unmatched gold entry. These are custom evidence-match diagnostics, not official CUAD metrics.


In [3]:
for lane, scores in result["splits"]["holdout"]["lanes"].items():
    for policy, score in scores.items():
        print(lane, policy, json.dumps(score["overall"]))


all legacy_mixed_units {"tp": 122, "fp": 42, "fn": 70, "precision": 0.7439024390243902, "recall": 0.6354166666666666, "f1": 0.6853932584269663}
all consistent_gold_entry_units {"tp": 122, "fp": 42, "fn": 208, "precision": 0.7439024390243902, "recall": 0.3696969696969697, "f1": 0.4939271255060729}
without_semantics_flags legacy_mixed_units {"tp": 72, "fp": 25, "fn": 113, "precision": 0.7422680412371134, "recall": 0.3891891891891892, "f1": 0.5106382978723404}
without_semantics_flags consistent_gold_entry_units {"tp": 72, "fp": 25, "fn": 258, "precision": 0.7422680412371134, "recall": 0.21818181818181817, "f1": 0.3372365339578454}


In [4]:
lane = result["splits"]["holdout"]["lanes"]["all"]["consistent_gold_entry_units"]
for category, counts in lane["per_category"].items():
    print(category, json.dumps(counts))
assert lane["overall"]["tp"] == 122
assert lane["overall"]["fp"] == 42
assert lane["overall"]["fn"] == 208
assert lane["overall"]["tp"] + lane["overall"]["fn"] == 330


Anti-Assignment {"tp": 15, "fp": 2, "fn": 38, "precision": 0.8823529411764706, "recall": 0.2830188679245283, "f1": 0.42857142857142855}
Audit Rights {"tp": 26, "fp": 6, "fn": 41, "precision": 0.8125, "recall": 0.3880597014925373, "f1": 0.5252525252525253}
Cap On Liability {"tp": 19, "fp": 3, "fn": 63, "precision": 0.8636363636363636, "recall": 0.23170731707317074, "f1": 0.36538461538461536}
Exclusivity {"tp": 23, "fp": 19, "fn": 18, "precision": 0.5476190476190477, "recall": 0.5609756097560976, "f1": 0.5542168674698795}
Governing Law {"tp": 27, "fp": 3, "fn": 8, "precision": 0.9, "recall": 0.7714285714285715, "f1": 0.8307692307692308}
Ip Ownership Assignment {"tp": 2, "fp": 7, "fn": 15, "precision": 0.2222222222222222, "recall": 0.11764705882352941, "f1": 0.15384615384615385}
Non-Compete {"tp": 5, "fp": 1, "fn": 12, "precision": 0.8333333333333334, "recall": 0.29411764705882354, "f1": 0.43478260869565216}
Termination For Convenience {"tp": 5, "fp": 1, "fn": 13, "precision": 0.833333333

## Attribution and limits

Source: CUAD v1, The Atticus Project (Hendrycks, Burns, Chen, Ball; 2021), CC BY 4.0. See README.md and provenance.json for pinned archive, selected IDs, modifications, matching policy and original execution gaps. The holdout was restarted after a runtime defect. These diagnostics do not establish a causal effect of governance versus model capability.
